# 15.3 Content-based & hybrid recommenders

When behavior is sparse, item attributes let us recommend by what something is, not only by who clicked it.

This gap topic adds side information to collaborative signals. Feature vectors represent item attributes, cosine scores a user profile against candidate content, and calibrated blending keeps cold items eligible. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1500
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)


def ndcg_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    gains = (2.0 ** relevance[order] - 1.0) / np.log2(np.arange(2, len(order) + 2))
    ideal = np.argsort(-relevance, kind="mergesort")[:k]
    ideal_gains = (2.0 ** relevance[ideal] - 1.0) / np.log2(np.arange(2, len(ideal) + 2))
    denom = np.sum(ideal_gains)
    if denom == 0.0:
        return 0.0
    return float(np.sum(gains) / denom)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def recall_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float) > 0.0
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    denom = np.sum(relevance)
    if denom == 0:
        return 0.0
    return float(np.sum(relevance[order]) / denom)


def rating_ladder():
    d1 = np.array([
        [5.0, 4.0, 0.0, 1.0],
        [4.0, 5.0, 0.0, 1.0],
        [1.0, 1.0, 5.0, 4.0],
        [0.0, 1.0, 4.0, 5.0],
    ])
    d2 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0],
        [4.0, 5.0, 0.0, 1.0, 2.0],
        [1.0, 1.0, 5.0, 4.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0],
        [5.0, 0.0, 1.0, 0.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0],
    ])
    d3 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0, 2.0],
        [4.0, 4.0, 0.0, 0.0, 2.0, 0.0],
        [1.0, 1.0, 5.0, 4.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 3.0, 0.0, 3.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 1.0],
    ])
    d4 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0],
        [4.0, 5.0, 0.0, 2.0, 0.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [1.0, 0.0, 4.0, 5.0, 0.0, 5.0, 0.0, 2.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0],
        [0.0, 3.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 1.0, 4.0, 0.0],
        [0.0, 4.0, 0.0, 5.0, 0.0, 5.0, 0.0, 3.0],
        [4.0, 0.0, 2.0, 0.0, 5.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 4.0, 5.0, 0.0, 4.0, 0.0, 0.0],
    ])
    d5 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [4.0, 5.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 0.0, 5.0, 0.0, 0.0, 0.0, 2.0],
        [0.0, 0.0, 5.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 1.0, 0.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 5.0, 0.0, 3.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 4.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 5.0],
    ])
    return [
        {"name": "D1 tiny rating matrix", "ratings": d1, "holdout": [(0, 2, 5.0)]},
        {"name": "D2 synthetic preferences", "ratings": d2, "holdout": [(0, 2, 5.0), (2, 4, 2.0), (5, 4, 3.0)]},
        {"name": "D3 noise ties sparsity", "ratings": d3, "holdout": [(0, 2, 5.0), (1, 3, 2.0), (6, 1, 3.0), (7, 4, 2.0)]},
        {"name": "D4 inline MovieLens-like sample", "ratings": d4, "holdout": [(0, 2, 4.0), (1, 5, 3.0), (4, 3, 2.0), (6, 7, 2.0)]},
        {"name": "D5 long-tail cold-start", "ratings": d5, "holdout": [(0, 2, 4.0), (2, 9, 2.0), (5, 8, 4.0), (10, 0, 3.0), (11, 1, 3.0)]},
    ]


def slate_ladder():
    return [
        {"name": "D1 3-item slate", "features": np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.744, 0.643, 0.819]), "collab": np.array([0.30, 0.90, 0.70]), "relevance": np.array([2.0, 3.0, 1.0])},
        {"name": "D2 synthetic preferences", "features": np.array([[1.0, 0.1, 0.0], [0.2, 0.9, 0.8], [0.8, 0.7, 0.1], [0.1, 0.2, 1.0]]), "content": np.array([0.70, 0.62, 0.88, 0.55]), "collab": np.array([0.40, 0.85, 0.65, 0.30]), "relevance": np.array([1.0, 3.0, 2.0, 0.0])},
        {"name": "D3 noise ties sparsity", "features": np.array([[1.0, 0.0, 0.2], [0.9, 0.2, 0.0], [0.1, 1.0, 0.9], [0.2, 0.8, 0.8], [0.0, 0.1, 1.0]]), "content": np.array([0.78, 0.78, 0.59, 0.60, 0.51]), "collab": np.array([0.20, 0.50, 0.88, 0.40, 0.10]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 0.0])},
        {"name": "D4 inline MovieLens-like ratings", "features": np.array([[1.0, 0.3, 0.0], [0.9, 0.4, 0.1], [0.2, 0.9, 0.6], [0.1, 0.6, 1.0], [0.7, 0.2, 0.8], [0.0, 0.1, 0.9]]), "content": np.array([0.82, 0.80, 0.66, 0.59, 0.75, 0.50]), "collab": np.array([0.55, 0.61, 0.92, 0.72, 0.35, 0.25]), "relevance": np.array([2.0, 2.0, 3.0, 1.0, 2.0, 0.0])},
        {"name": "D5 long-tail cold-start", "features": np.array([[1.0, 0.2, 0.0], [0.8, 0.1, 0.1], [0.1, 0.9, 0.8], [0.0, 0.7, 1.0], [0.9, 0.9, 0.0], [0.1, 0.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.84, 0.78, 0.61, 0.58, 0.90, 0.52, 0.82]), "collab": np.array([0.65, 0.20, 0.91, 0.55, np.nan, 0.05, np.nan]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 3.0, 0.0, 2.0])},
    ]


def print_ladder_preview(rungs):
    for rung in rungs:
        if "ratings" in rung:
            matrix = rung["ratings"]
            observed = int(np.sum(matrix > 0.0))
            total = int(matrix.size)
            print(rung["name"], matrix.shape, "observed", observed, "of", total)
            print(matrix[:3, :5])
        else:
            relevance = rung["relevance"]
            print(rung["name"], "items", len(relevance), "positives", int(np.sum(relevance > 0.0)))
            print("relevance", relevance[:5])


## The concept, built once: profile plus hybrid blend

A rating-weighted content profile is $p=rac{\sum_i r_i x_i}{\sum_i r_i}$. The hybrid score is $$s_{hybrid}(u,i)=lpha s_{content}(u,i)+(1-lpha)s_{collab}(u,i).$$ Missing collaboration for a new item is unknown, not zero.

In [ ]:

def content_profile(features, ratings):
    features = np.asarray(features, dtype=float)
    ratings = np.asarray(ratings, dtype=float)
    mask = ratings > 0.0
    weighted = ratings[mask, None] * features[mask]
    return np.sum(weighted, axis=0) / np.sum(ratings[mask])


def content_scores(profile, candidates):
    return np.array([cosine(profile, candidate) for candidate in candidates])


def hybrid_score(content, collab, alpha=0.4, fallback_to_content=True):
    content = np.asarray(content, dtype=float)
    collab = np.asarray(collab, dtype=float)
    if fallback_to_content:
        collab = np.where(np.isnan(collab), content, collab)
    else:
        collab = np.where(np.isnan(collab), 0.0, collab)
    return alpha * content + (1.0 - alpha) * collab


## Check the lesson numbers

The lesson profile is $p=(0.900,0.500,0.600)$. Candidate cosines are $0.744$, $0.643$, and $0.819$; with $lpha=0.4$, hybrid scores are $0.496$, $0.836$, and $0.760$.

In [ ]:

features = np.array([
    [1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0],
    [0.0, 0.0, 1.0],
])
ratings = np.array([5.0, 4.0, 1.0, 0.0])
profile = content_profile(features, ratings)
candidates = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
])
cos_scores = content_scores(profile, candidates)
hybrid = hybrid_score(np.array([0.79, 0.74, 0.85]), np.array([0.30, 0.90, 0.70]), alpha=0.4)

assert np.allclose(np.round(profile, 3), np.array([0.900, 0.500, 0.600]))
assert np.allclose(np.round(cos_scores, 3), np.array([0.744, 0.643, 0.819]))
assert np.allclose(np.round(hybrid, 3), np.array([0.496, 0.836, 0.760]))

print("profile", np.round(profile, 3))
print("cosines", np.round(cos_scores, 3))
print("hybrid", np.round(hybrid, 3))


## The dataset ladder

D1 is the lesson slate, D2 adds synthetic preferences, D3 adds noise and ties, D4 is an inline ratings-style slate, and D5 adds long-tail/cold-start candidates.

In [ ]:

rungs = slate_ladder()
print_ladder_preview(rungs)


In [ ]:

rows = []
for rung in rungs:
    scores = hybrid_score(rung["content"], rung["collab"], alpha=0.4, fallback_to_content=True)
    metric = ndcg_at_k(rung["relevance"], scores, k=3)
    rows.append({"name": rung["name"], "scores": scores, "metric": metric, "relevance": rung["relevance"]})

for row in rows:
    print(f"{row['name']}: NDCG@3={row['metric']:.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
last = rows[-1]
order = np.argsort(-last["scores"])
axes[0].bar(np.arange(len(order)), last["scores"][order], color="steelblue")
axes[0].set_title("D5 ranked hybrid slate")
axes[0].set_xlabel("rank")
axes[0].set_ylabel("hybrid score")
axes[1].plot([row["name"].split()[0] for row in rows], [row["metric"] for row in rows], marker="o")
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title("NDCG@3 vs sparsity rung")
axes[1].set_ylabel("NDCG@3")
fig.tight_layout()


## Pitfall on D5: over-specialized profiles and missing collaboration

A narrow content-only profile can form a bubble, and treating absent collaborative scores as zero punishes cold inventory. The fix calibrates the blend and falls back to content for unknown collaboration.

In [ ]:

d5 = slate_ladder()[-1]
wrong_scores = hybrid_score(d5["content"], d5["collab"], alpha=0.2, fallback_to_content=False)
fixed_scores = hybrid_score(d5["content"], d5["collab"], alpha=0.5, fallback_to_content=True)
wrong_metric = ndcg_at_k(d5["relevance"], wrong_scores, k=3)
fixed_metric = ndcg_at_k(d5["relevance"], fixed_scores, k=3)

print("wrong cold-item-as-zero NDCG@3", round(wrong_metric, 3))
print("fixed calibrated fallback NDCG@3", round(fixed_metric, 3))
print("cold item fixed score", round(fixed_scores[4], 3))


## Evaluate it + Practice

Evaluation checklist:
- Track `NDCG@3` on D1--D5 and compare with a no-skill baseline such as popularity or original order.
- Run a sanity check where the strongest observed signal is swapped and the top item changes.
- Ablate the key idea, such as masking, latent factors, calibration, or query-local losses, and confirm the metric drops.
- Watch failure signals: identical recommendations for everyone, head-item dominance, unstable tiny-overlap scores, and poor cold-start behavior.

Practice:
1. Change one D3 tie and predict which slate position moves before running the cell.


2. Change `alpha` from 0.1 to 0.9 and plot the winner.
3. Remove the cold item fallback and explain which item disappears.